# Task
Create a synthetic data generator for business use cases (such as testing HR systems) using the "meta-llama/Llama-3.2-3B-Instruct" model.

First, initialize the model with `BitsAndBytesConfig` 4-bit quantization (NF4) for efficient execution. Then, implement a function to generate structured synthetic records (e.g., employee data) and parse them into a Pandas DataFrame. Finally, build a Gradio interface that allows users to select a data category and the number of records, displaying the result in a table for export.




## Define Data Strategy

### Subtask:
Outline the data schema, format, and business purpose for the synthetic data generator.


### Synthetic Data Strategy: HR System Testing

#### 1. Business Purpose
The primary goal is to generate high-quality synthetic HR data for testing human resource management systems (HRIS). This allows for performance testing, UI development, and analytics prototyping without exposing sensitive Personal Identifiable Information (PII).

#### 2. Data Schema (Employee Record)
Each generated record will represent an 'Employee' with the following fields:
- **Employee ID**: Unique identifier (e.g., EMP-001).
- **Full Name**: Randomly generated realistic names.
- **Department**: One of: Engineering, Sales, Marketing, HR, Finance, or Legal.
- **Job Title**: Role-appropriate titles (e.g., Software Engineer, Account Manager).
- **Salary**: Numerical value within a realistic range ($40,000 - $200,000).
- **Hire Date**: Dates ranging from 2010 to the present.
- **Performance Rating**: Categorical value (1-5 or Excellent, Good, Average, etc.).

#### 3. Output Format
To ensure compatibility with downstream analysis, the model must output data in **JSON** or **CSV** format. This allows for seamless loading into a `pandas.DataFrame` for further processing.

#### 4. Constraints & Distributions
- **Realistic Salaries**: Salaries should be roughly correlated with Job Titles and seniority.
- **Departmental Balance**: Records should be distributed across departments to avoid bias unless specified.
- **Uniqueness**: Employee IDs must be unique across the dataset.

## Setup Quantized Model

Initialize the Llama-3.2-3B-Instruct model using BitsAndBytes 4-bit quantization (NF4) for efficient execution in Colab.


In [ ]:
!uv install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from google.colab import userdata
from huggingface_hub import login

# 1. Define Model ID
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# 2. Configure BitsAndBytes for 4-bit NF4 quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. Retrieve Token and Login
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=True)

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

# 5. Load Quantized Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token
)

print(f"Model {MODEL_ID} and tokenizer loaded successfully with 4-bit NF4 quantization.")

## Implement Generator Logic

Create a function to generate synthetic records using the quantized Llama model and parse the results into a Pandas DataFrame.


In [2]:
import json
import re
import pandas as pd

def generate_synthetic_data(category, num_records):
    """
    Generates synthetic records using the Llama model and parses them into a DataFrame.
    """
    # 1. Construct the prompt based on the HR strategy
    prompt = f"""Generate a valid JSON list containing {num_records} synthetic {category} records.
Each record must strictly follow this schema:
- Employee ID: Unique string (e.g., EMP-001)
- Full Name: Realistic name
- Department: Engineering, Sales, Marketing, HR, Finance, or Legal
- Job Title: Appropriate for the department
- Salary: Integer between 40000 and 200000
- Hire Date: YYYY-MM-DD between 2010 and 2024
- Performance Rating: Integer 1-5

Return ONLY the JSON list. Do not include any explanations or markdown code blocks."""

    # 2. Tokenize and generate
    messages = [{"role": "user", "content": prompt}]
    # Extract 'input_ids' tensor from the BatchEncoding object returned by apply_chat_template
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")['input_ids'].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=1000,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    # 3. Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 4. Extract JSON content using regex (finding the content between [])
    try:
        json_match = re.search(r'\[.*\d.*\]', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            data = json.loads(json_str)
        else:
            # Fallback to direct load if regex fails but response is raw JSON
            data = json.loads(response)

        # 5. Convert to DataFrame
        df = pd.DataFrame(data)
        return df
    except Exception as e:
        print(f"Error parsing model output: {e}")
        print("Raw Response:", response)
        return pd.DataFrame()

# Test the function with 3 records
test_df = generate_synthetic_data('HR/Employee', 3)
print("Generated Synthetic Data:")
display(test_df)

And to save it into a Google sheet should be so easy

In [1]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=test_df)